<a href="https://colab.research.google.com/github/aligreo/LLM-Finetuning/blob/main/lora_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!uv pip install bitsandbytes trl

In [1]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch
print(torch.cuda.get_device_name(0))
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Tesla T4
Memory: 15.6 GB


In [2]:
import os
import torch, time, random, json, re
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    default_data_collator,
    EarlyStoppingCallback,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType

try:
    from trl import SFTTrainer, SFTConfig
except ImportError:
    !pip install -q trl
    from trl import SFTTrainer, SFTConfig

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [3]:
class Config:
    SEED = 42
    MODEL_NAME = "mistralai/Mistral-7B-v0.1"

    # Dataset Sizes
    TRAIN_SIZE = 2000
    TEST_SIZE = 200
    GEN_SIZE = 100
    FORGET_SIZE = 100

    # Training Params
    MAX_SEQ_LEN = 512
    CONTEXT_LEN = 800
    EPOCHS = 3
    BATCH_SIZE = 4
    LR = 2e-4

    # LoRA Params
    LORA_R = 16
    LORA_ALPHA = 32
    LORA_DROPOUT = 0.1

    # Paths
    OUTPUT_DIR = "./lora_model"
    RESULTS_FILE = "results.json"

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(Config.SEED)

In [4]:
def load_squad_splits():
    print("Loading SQuAD Dataset splits...")
    return {
        'train': load_dataset("squad", split=f"train[:{Config.TRAIN_SIZE}]"),
        'test': load_dataset("squad", split=f"validation[:{Config.TEST_SIZE}]"),
        'gen': load_dataset("squad", split=f"validation[{Config.TEST_SIZE}:{Config.TEST_SIZE + Config.GEN_SIZE}]"),
        'forget': load_dataset("squad", split=f"validation[{Config.TEST_SIZE + Config.GEN_SIZE}:{Config.TEST_SIZE + Config.GEN_SIZE + Config.FORGET_SIZE}]")
    }

datasets = load_squad_splits()
for k, v in datasets.items():
    print(f"{k.capitalize():<10}: {len(v)} samples")

Loading SQuAD Dataset splits...
Train     : 2000 samples
Test      : 200 samples
Gen       : 100 samples
Forget    : 100 samples


In [5]:
tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(example):
    prompt = f"Context: {example['context'][:Config.CONTEXT_LEN]}\nQuestion: {example['question']}\nAnswer: "
    full_text = prompt + example['answers']['text'][0]

    tokens = tokenizer(full_text, truncation=True, max_length=Config.MAX_SEQ_LEN, padding="max_length")
    prompt_len = len(tokenizer(prompt, truncation=True, max_length=Config.MAX_SEQ_LEN)['input_ids'])

    labels = ([-100] * prompt_len + tokens["input_ids"][prompt_len:])[:Config.MAX_SEQ_LEN]
    labels += [-100] * (Config.MAX_SEQ_LEN - len(labels))
    tokens["labels"] = labels
    return tokens

def normalize_text(text):
    text = text.strip().lower()
    text = re.sub(r'\b(a|an|the)\b', ' ', text)
    text = re.sub(r'[^\w\s]', '', text)
    return re.sub(r'\s+', ' ', text).strip()

def get_metrics(pred, gold):
    pred, gold = normalize_text(pred), normalize_text(gold)
    em = 1 if pred == gold else 0
    p_tokens, g_tokens = pred.split(), gold.split()
    if not p_tokens or not g_tokens: return em, 0.0
    common = set(p_tokens) & set(g_tokens)
    if not common: return em, 0.0
    prec, rec = len(common)/len(p_tokens), len(common)/len(g_tokens)
    f1 = 2 * prec * rec / (prec + rec)
    return em, f1

def run_evaluation(model, dataset, name="Eval"):
    model.eval()
    results = {'em': 0, 'f1': 0, 'acc': 0, 'time': 0}
    print(f"\nEvaluating {name}...")

    for i, item in enumerate(dataset):
        prompt = f"Context: {item['context'][:Config.CONTEXT_LEN]}\nQuestion: {item['question']}\nAnswer:"
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=Config.MAX_SEQ_LEN).to("cuda")

        t0 = time.time()
        with torch.no_grad():
            # Explicitly set pad_token_id to suppress warnings
            out = model.generate(
                **inputs,
                max_new_tokens=30,
                do_sample=False,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.pad_token_id
            )
        results['time'] += (time.time() - t0)

        gen = tokenizer.decode(out[0], skip_special_tokens=True)
        pred = gen.split("Answer:")[-1].split("\n")[0].strip().lower()
        gold = item['answers']['text'][0].lower()

        em, f1 = get_metrics(pred, gold)
        results['em'] += em
        results['f1'] += f1
        if gold in pred or pred in gold: results['acc'] += 1

    count = len(dataset)
    return {k: (v/count)*100 if k != 'time' else v/count for k, v in results.items()}

train_tokenized = datasets['train'].map(tokenize_fn, remove_columns=datasets['train'].column_names)
test_tokenized = datasets['test'].map(tokenize_fn, remove_columns=datasets['test'].column_names)

In [ ]:
print("=== STEP 3: BASE MODEL EVALUATION ===")

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    Config.MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto"
)

base_results = {
    'main': run_evaluation(base_model, datasets['test'], "Base-Main"),
    'gen': run_evaluation(base_model, datasets['gen'], "Base-Generalization"),
    'forget': run_evaluation(base_model, datasets['forget'], "Base-Forgetting")
}

del base_model
torch.cuda.empty_cache()

=== STEP 3: BASE MODEL EVALUATION ===


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]


Evaluating Base-Main...

Evaluating Base-Generalization...


In [ ]:
print("=== STEP 4: LoRA INITIALIZATION ===")

# Reuse quant_config from previous step
model = AutoModelForCausalLM.from_pretrained(
    Config.MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto"
)

lora_config = LoraConfig(
    r=Config.LORA_R,
    lora_alpha=Config.LORA_ALPHA,
    lora_dropout=Config.LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()

In [10]:
def formatting_prompts_func(example):
    output_texts = []
    for i in range(len(example['question'])):
        text = f"Context: {example['context'][i][:Config.CONTEXT_LEN]}\nQuestion: {example['question'][i]}\nAnswer: {example['answers'][i]['text'][0]}"
        output_texts.append(text)
    return output_texts

trainer = SFTTrainer(
    model=lora_model,
    train_dataset=datasets['train'],
    eval_dataset=datasets['test'],
    peft_config=lora_config,
    formatting_func=formatting_prompts_func,
    data_collator=default_data_collator,
    args=SFTConfig(
        output_dir=Config.OUTPUT_DIR,
        num_train_epochs=Config.EPOCHS,
        per_device_train_batch_size=Config.BATCH_SIZE,
        learning_rate=Config.LR,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        fp16=True,
        report_to="none",
        seed=Config.SEED,
        max_seq_length=Config.MAX_SEQ_LEN,
        dataset_text_field=None # We use formatting_func instead
    ),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

start_time = time.time()
trainer.train()
train_duration = (time.time() - start_time) / 60
lora_model.save_pretrained(Config.OUTPUT_DIR)

NameError: name 'lora_model' is not defined

In [ ]:
print("=== STEP 7: LoRA EVALUATION ===")
lora_results = {
    'main': run_evaluation(lora_model, datasets['test'], "LoRA-Main"),
    'gen': run_evaluation(lora_model, datasets['gen'], "LoRA-Generalization"),
    'forget': run_evaluation(lora_model, datasets['forget'], "LoRA-Forgetting")
}

In [ ]:
print("\n" + "="*60)
print("FINAL REFACTORED SUMMARY")
print("="*60)
print(f"Base Acc: {base_results['main']['acc']:.1f}% | LoRA Acc: {lora_results['main']['acc']:.1f}%")
print(f"Base F1 : {base_results['main']['f1']:.1f}% | LoRA F1 : {lora_results['main']['f1']:.1f}%")
print(f"Generalization Delta: {lora_results['gen']['acc'] - base_results['gen']['acc']:+.1f}%")
print(f"Forgetting Delta: {lora_results['forget']['acc'] - base_results['forget']['acc']:+.1f}%")
print(f"Training Time: {train_duration:.1f} min")

with open(Config.RESULTS_FILE, "w") as f:
    json.dump({'base': base_results, 'lora': lora_results}, f, indent=2)